# Retail Sales Analysis & Forecasting
This notebook analyzes retail sales, profit, and a simple monthly forecast.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().resolve().parents[0] if 'notebooks' in str(Path.cwd()) else Path.cwd()
DATA = ROOT / 'data' / 'raw' / 'retail_sales.csv'
df = pd.read_csv(DATA, parse_dates=['order_date'])
df.head()

In [ ]:
category_summary = df.groupby('product_category', as_index=False).agg(total_sales=('sales','sum'), total_profit=('profit','sum')).sort_values('total_sales', ascending=False)
category_summary

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(category_summary['product_category'], category_summary['total_sales'])
plt.title('Sales by Category')
plt.xlabel('Category')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()

In [ ]:
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False).agg(monthly_sales=('sales','sum'))
monthly['month_start'] = pd.to_datetime(monthly['month'] + '-01')
monthly['t'] = np.arange(len(monthly))
train = monthly.iloc[:-6].copy()
test = monthly.iloc[-6:].copy()
model = LinearRegression().fit(train[['t']], train['monthly_sales'])
test['pred'] = model.predict(test[['t']])
mean_absolute_error(test['monthly_sales'], test['pred'])

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(monthly['month_start'], monthly['monthly_sales'])
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

## Next step
Run `python src/analyze_retail_sales.py` from the project root to export cleaned tables, charts, and forecast metrics.